In [1]:
# !pip install langchain_huggingface
# !pip install sentence-transformers

In [ ]:
from __future__ import annotations

from typing import List, Dict, Any, Optional
import os
import asyncio

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from src.faiss_storage import FaissStorage


EMB_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMB_DIM = 384
CHUNK_SIZE=512
CHUNK_OVERLAP=200

embed_model = HuggingFaceEmbeddings(model_name=EMB_MODEL_NAME)
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


# load the pdf file from the path and chunk it
def load_and_chunk_pdf(pdf_path: str) -> List[Document]:
    loader = PyPDFLoader(pdf_path)
    text = loader.load()
    text = [d.page_content for d in text if getattr(d, "page_content", None)]

    chunks = []
    for t in text:
        chunks.extend(splitter.split_text(t))
    
    return chunks



def embed_texts(texts: list[str]) -> list[list[float]]:
    response = embed_model.embed_documents(texts)
    return response

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
pdf_path = r"C:\\Users\\yashd\\OneDrive\\Desktop\\updated cvs\\UK resumes\\yash_jobster.pdf"

pages = load_and_chunk_pdf(pdf_path)
text_embs = embed_texts(pages)

store = FaissStorage().upsert(ids=[i for i in range(len(pages))], vectors=text_embs, payloads=[{"text": pages[i], "source": pdf_path} for i in range(len(pages))])  

In [8]:
from src.faiss_storage import FaissStorage
from src.ingest_pdf import embed_texts
store = FaissStorage()

query = "who is yash"
text_emb = embed_texts(query)[0]

result = store.search(
    query_vector=text_emb,
    top_k=5
)

result

{'contexts': ['Yash Dhakade \nGlasgow, UK \n+44 7823703834, \nyashdhakade2021@gmail.com \nwww.linkedin.com/in/yashdhakade \nSUMMARY \nApplied AI Engineer with hands-on experience building and deploying Python-based ML and LLM systems used in \nreal-world workflows. Strong background across the full AI lifecycle, including data pipeline design, model training and \nevaluation, API-backed services, and deployment-ready ML systems. Experienced with Retrieval-Augmented',
  'analysis, feature engineering \nGenerative AI & LLMs: Retrieval-Augmented Generation (RAG), LangChain, embeddings, prompt pipelines, \nbias evaluation \nMLOps & Deployment: Docker, AWS (EC2, S3), model evaluation, experiment tracking, reproducible \npipelines \nData & Search Systems: FAISS, vector databases, large-scale document retrieval',
  'Quantitative Methods for Artificial Intelligence. \nDissertation: Experimental evaluation of NLP and Large Language Model systems, with emphasis on \nbias detection, robustness an